# Astra v3 — Stage 2: Fine-Tuning on Held-Out test21

Fine-tunes the pre-trained cyber model on the **held-out test21 dataset** (11.8k rows, 17 novel attack classes never seen during pre-training).

📂 **No Google Drive used** (Google Drive space is full) — checkpoints are saved locally on the Colab VM and auto-downloaded to your computer.

Runtime: **Settings → Runtime → GPU (T4/H100)**, Internet ON. Run cells top to bottom.

In [ ]:
import torch
print('torch', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))

In [ ]:
import os, subprocess, shutil, json

REPO = '/content/astra'
GIT_URL = 'https://github.com/anoneurx/astra.git'

if os.path.isdir(REPO):
    subprocess.run(f'rm -rf {REPO}', shell=True, check=False)
subprocess.run(f'git clone --depth 1 {GIT_URL} {REPO}', shell=True, check=True)
assert os.path.isdir(REPO), 'clone failed'
os.chdir(REPO)
print('repo ready at', os.getcwd())

**Upload Pre-Train Checkpoint** (`astra_cyber_pretrain_final.npz`):
Since Google Drive is full, upload your pre-trained model checkpoint directly from your computer (`/run/media/kashie/28df1a2c-ca37-4ab9-acc0-159d03b34417/Project Astra/cyberv3` or wherever you saved it).

In [ ]:
from google.colab import files
import os

os.makedirs('/content/checkpoints', exist_ok=True)
print('Please upload your astra_cyber_pretrain_final.npz checkpoint file:')
uploaded = files.upload()

pretrain_path = None
for fn in uploaded.keys():
    pretrain_path = os.path.join('/content/checkpoints', fn)
    os.rename(fn, pretrain_path)
    print(f'Saved uploaded checkpoint to {pretrain_path} ({os.path.getsize(pretrain_path):,} bytes)')

assert pretrain_path and os.path.exists(pretrain_path), 'Checkpoint upload failed or missing!'

---
## Build cyber BPE tokenizer

In [ ]:
os.chdir('/content/astra')
if not os.path.exists('tokenizer/artifacts/cyber_bpe.json'):
    subprocess.run('python tokenizer/train_tokenizer.py --config configs/tokenizer_cyber.json', shell=True, check=True)
print('Tokenizer ready')

---
## STAGE 2 — Fine-tune on held-out test21 (10,000 steps, ~10 min)
Warm-starts from the uploaded pre-train checkpoint and fine-tunes on `datasets/cyber/test21.txt`.

In [ ]:
import os, subprocess, shutil, json
os.chdir('/content/astra')

# Create fine-tune config overriding train data to test21
cfg = json.load(open('configs/astra5m_cyber.json'))
cfg['data']['train'] = 'datasets/cyber/test21.txt'
cfg['data']['val'] = 'datasets/cyber/val.txt'
cfg['training']['max_steps'] = 10000
cfg['training']['val_every'] = 500
cfg['out_dir'] = 'checkpoints/astra5m_cyber_finetune'

ft_cfg_path = '/content/astra5m_cyber_finetune.json'
json.dump(cfg, open(ft_cfg_path, 'w'), indent=2)
print('Fine-tune config written to', ft_cfg_path)

!mkdir -p /content/runs/cyber_finetune

# Run training warm-starting from uploaded pre-train checkpoint
res = subprocess.run(
    f'python training/gpu_train.py --config {ft_cfg_path} --steps 10000 --out /content/runs/cyber_finetune --resume {pretrain_path} --reset-step --cache-dir /content/cache',
    shell=True, capture_output=True, text=True
)
print('STDOUT:\n', res.stdout[-3000:])
print('STDERR:\n', res.stderr[-3000:])
if res.returncode != 0:
    raise RuntimeError('Fine-tuning failed — see STDERR above')

SRC = '/content/runs/cyber_finetune/resumed/final.npz'
assert os.path.exists(SRC), 'FINETUNE FINAL NOT PRODUCED'

dest = '/content/astra_cyber_finetune_final.npz'
shutil.copy(SRC, dest)
print('FINETUNE ok:', os.path.getsize(dest), 'bytes')

from google.colab import files
files.download(dest)
print('✅ Download started for astra_cyber_finetune_final.npz')